# Problem Book P3, Chapter 1: Sums of Random Pieces (Python companion)

Part A checks the numbers of the chapter, problem by problem: exact enumerations where they are possible, simulations where they are not.
Part B is a laboratory with three experiments that push the problems further, exercises first and solutions at the end.
We use `numpy`, `mpmath` and `sympy`; the notebook runs in a few minutes.

*Before running a cell, predict its output.*

In [1]:
import math, random, itertools
from math import comb, sqrt, pi, exp, log, factorial
from fractions import Fraction as Fr
import numpy as np
import mpmath as mp
from sympy import primerange, bell
from sympy.functions.combinatorial.numbers import stirling
rng = np.random.default_rng(2026)
random.seed(2026)
phi = (1 + sqrt(5)) / 2

## Part A: checks

**Problem 1.1 (pairwise independent signs).** With $k$ fair signs, the $2^k-1$ products $\chi_S$ are pairwise independent, and their sum is
$\prod(1+\varepsilon_i)-1\in\{-1,2^k-1\}$. Check both facts for $k=4$ by listing the $16$ outcomes.

In [2]:
k = 4
subsets = [S for r in range(1, k + 1) for S in itertools.combinations(range(k), r)]
outcomes = list(itertools.product([-1, 1], repeat=k))
chi = lambda S, e: math.prod(e[i] for i in S)
ok = all(sum(chi(S, e) * chi(T, e) for e in outcomes) == 0 for S in subsets for T in subsets if S != T)
print("pairwise uncorrelated (hence independent for signs):", ok, "  mean zero:", all(sum(chi(S, e) for e in outcomes) == 0 for S in subsets))
print("values of the sum:", sorted(set(sum(chi(S, e) for S in subsets) for e in outcomes)))

pairwise uncorrelated (hence independent for signs): True   mean zero: True
values of the sum: [-1, 15]


**Problem 1.2 (Kac's sawtooth).** $\int_0^1B(x)B(mx)\,dx=\frac1{12m}$ for the sawtooth $B(x)=\{x\}-\frac12$, so the variance of
$\frac1{\sqrt n}\sum_{k<n}B(2^kx)$ tends to $\frac1{12}+2\sum_{k\ge1}\frac1{12\cdot2^k}=\frac14$, and with $3^k$ to $\frac16$. Check by exact integration and by simulation.

In [3]:
def corr(m, N=200000):
    x = (np.arange(N) + 0.5) / N
    return np.mean((x - 0.5) * ((m * x) % 1 - 0.5))
for m in (1, 2, 3, 4, 8):
    print("m =", m, " integral", round(corr(m), 7), " 1/(12m) =", round(1 / (12 * m), 7))
def sim(base, n=40, reps=200000):
    # x is drawn with 64 random base-b digits; the k-th term only needs digits k+1, k+2, ...
    D = rng.integers(0, base, size=(reps, n + 30))
    w = base ** -np.arange(1, 31, dtype=float)
    S = sum(D[:, k:k + 30] @ w - 0.5 for k in range(n))
    return S.var() / n
print("variance per term, base 2:", round(sim(2), 4), " (limit 1/4, finite-n value", round(0.25 - (2/3) * (1 - 2.0**-40) / 40 * 0.5, 4), ")")
print("variance per term, base 3:", round(sim(3), 4), " (limit 1/6)")

m = 1  integral 0.0833333  1/(12m) = 0.0833333
m = 2  integral 0.0416667  1/(12m) = 0.0416667
m = 3  integral 0.0277772  1/(12m) = 0.0277778
m = 4  integral 0.0208333  1/(12m) = 0.0208333
m = 8  integral 0.0104167  1/(12m) = 0.0104167


variance per term, base 2: 0.2415  (limit 1/4, finite-n value 0.2417 )


variance per term, base 3: 0.1644  (limit 1/6)


**Problem 1.3 (peaks).** Exact mean $\frac{n-2}3$ and variance $\frac{2(n+1)}{45}$ for $n\ge4$, by listing all permutations; and the up-down count $16$ of
patterns $a<b>c<d>e$.

In [4]:
for n in range(4, 10):
    vals = [sum(1 for i in range(1, n - 1) if p[i - 1] < p[i] > p[i + 1]) for p in itertools.permutations(range(n))]
    N = len(vals); m = Fr(sum(vals), N); v = Fr(sum(x * x for x in vals), N) - m * m
    print(n, m, v, v == Fr(2 * (n + 1), 45))
print("permutations of 5 with a<b>c<d>e:", sum(1 for p in itertools.permutations(range(5)) if p[0] < p[1] > p[2] < p[3] > p[4]))

4 2/3 2/9 True
5 1 4/15 True
6 4/3 14/45 True
7 5/3 16/45 True
8 2 2/5 True


9 7/3 4/9 True
permutations of 5 with a<b>c<d>e: 16


**Problem 1.4 (random tilings).** $P(D=k)=\binom{n-k}k/F_{n+1}$; the mean is $\frac{5-\sqrt5}{10}n-\frac{\sqrt5-1}{10}+o(1)$ and the variance $\sim\frac n{5\sqrt5}$.

In [5]:
for n in (10, 100, 1000, 5000):
    w = [comb(n - k, k) for k in range(n // 2 + 1)]; T = sum(w)
    m = Fr(sum(k * x for k, x in enumerate(w)), T); v = Fr(sum(k * k * x for k, x in enumerate(w)), T) - m * m
    print(n, " mean - (5-sqrt5)n/10 =", round(float(m) - (5 - sqrt(5)) * n / 10, 6), " -(sqrt5-1)/10 =", round(-(sqrt(5) - 1) / 10, 6),
          "  var/n =", round(float(v) / n, 6), " 1/(5 sqrt5) =", round(1 / (5 * sqrt(5)), 6))
q = 1 / phi ** 2
print("u_n = P(hit n) for the renewal walk, n = 1..8:", [round((1 - (-q) ** (n + 1)) / (1 + q), 6) for n in range(1, 9)], " limit 1/mu =", round(1 / (1 + q), 6))

10  mean - (5-sqrt5)n/10 = -0.123483  -(sqrt5-1)/10 = -0.123607   var/n = 0.090443  1/(5 sqrt5) = 0.089443
100  mean - (5-sqrt5)n/10 = -0.123607  -(sqrt5-1)/10 = -0.123607   var/n = 0.089537  1/(5 sqrt5) = 0.089443
1000  mean - (5-sqrt5)n/10 = -0.123607  -(sqrt5-1)/10 = -0.123607   var/n = 0.089452  1/(5 sqrt5) = 0.089443


5000  mean - (5-sqrt5)n/10 = -0.123607  -(sqrt5-1)/10 = -0.123607   var/n = 0.089445  1/(5 sqrt5) = 0.089443
u_n = P(hit n) for the renewal walk, n = 1..8: [0.618034, 0.763932, 0.708204, 0.72949, 0.72136, 0.724465, 0.723279, 0.723732]  limit 1/mu = 0.723607


**Problem 1.5 (Harper).** The blocks of a random set partition: mean and variance from the Stirling numbers, the identity
$\operatorname{Var}K=\operatorname{Var}M-1$ with Dobiński's $M$, real roots (checked with exact Sturm sequences), and the mode within $1$ of the mean (Darroch).

In [6]:
from sympy import Poly, symbols, sturm, Rational
X = symbols('x')
for n in (5, 10, 20, 30, 40):
    S = [stirling(n, k) for k in range(n + 1)]; B = [bell(n + j) for j in range(3)]
    m = Fr(sum(k * s for k, s in enumerate(S)), B[0]); v = Fr(sum(k * k * s for k, s in enumerate(S)), B[0]) - m * m
    VM = Fr(B[2], B[0]) - Fr(B[1], B[0]) ** 2
    P = Poly(sum(s * X ** k for k, s in enumerate(S)), X)
    nreal = P.count_roots()        # real roots with multiplicity
    mode = max(range(n + 1), key=lambda k: S[k])
    print(f"n={n:3d}  E K={float(m):8.4f}  Var K={float(v):7.4f}  Var M - 1={float(VM - 1):7.4f}  real roots {nreal}/{n}  mode {mode}  |mode-mean|<1: {abs(mode - m) < 1}")

n=  5  E K=  2.9038  Var K= 0.6254  Var M - 1= 0.6254  real roots 5/5  mode 3  |mode-mean|<1: True
n= 10  E K=  4.8510  Var K= 1.0977  Var M - 1= 1.0977  real roots 10/10  mode 5  |mode-mean|<1: True
n= 20  E K=  8.1808  Var K= 1.8425  Var M - 1= 1.8425  real roots 20/20  mode 8  |mode-mean|<1: True


n= 30  E K= 11.1563  Var K= 2.4665  Var M - 1= 2.4665  real roots 30/30  mode 11  |mode-mean|<1: True


n= 40  E K= 13.9326  Var K= 3.0244  Var M - 1= 3.0244  real roots 40/40  mode 14  |mode-mean|<1: True


**Problem 1.6 (fixed points).** Exact total variation distance between the number of fixed points and Poisson$(1)$, against the bound $\frac{2^n}{(n+1)!}$.

In [7]:
def fixed_law(n):
    return [Fr(1, factorial(k)) * sum(Fr((-1) ** j, factorial(j)) for j in range(n - k + 1)) for k in range(n + 1)]
for n in (3, 5, 8, 10, 15, 20):
    L = fixed_law(n)
    po = [mp.e ** -1 / mp.factorial(k) for k in range(n + 60)]
    tv = (sum(abs(mp.mpf(L[k].numerator) / L[k].denominator - po[k]) for k in range(n + 1)) + sum(po[n + 1:])) / 2
    print(n, mp.nstr(tv, 6), " bound 2^n/(n+1)! =", mp.nstr(mp.mpf(2) ** n / mp.factorial(n + 1), 6))

3 0.237474  bound 2^n/(n+1)! = 0.333333
5 0.0344083  bound 2^n/(n+1)! = 0.0444444
8 0.000586429  bound 2^n/(n+1)! = 0.000705467
10 2.19535e-5  bound 2^n/(n+1)! = 2.56534e-5
15 1.4004e-9  bound 2^n/(n+1)! = 1.56614e-9
20 1.88063e-14  bound 2^n/(n+1)! = 2.05237e-14


**Problem 1.7 (the Kubilius model converges slowly).** Stein's bound $d_W\le3\sum_p\mathbb E|X_p|^3$ and $d_K\le(2/\pi)^{1/4}\sqrt{d_W}$, against the exact
Kolmogorov distance of the model, computed by convolution, for $x=10^6$.

In [8]:
x = 10 ** 6
ps = list(primerange(2, x + 1))
sig2 = sum(1 / p * (1 - 1 / p) for p in ps); mu = sum(1 / p for p in ps)
third = sum((1 / p) * (1 - 1 / p) * (1 - 2 / p + 2 / p ** 2) for p in ps)
dW = 3 * third / sig2 ** 1.5
print("loglog x =", round(log(log(x)), 4), " mean =", round(mu, 4), " variance =", round(sig2, 4))
print("Stein: d_W <=", round(dW, 4), "  d_K <=", round((2 / pi) ** 0.25 * sqrt(dW), 4))
law = np.zeros(80); law[0] = 1.0
for p in ps:
    law[1:] = law[1:] * (1 - 1 / p) + law[:-1] / p; law[0] *= (1 - 1 / p)
cdf = np.cumsum(law)
from math import erf
Phi = lambda t: 0.5 * (1 + erf(t / sqrt(2)))
dK = max(max(abs(cdf[k] - Phi((k - mu) / sqrt(sig2))), abs((cdf[k - 1] if k else 0) - Phi((k - mu) / sqrt(sig2)))) for k in range(80))
print("exact Kolmogorov distance of the model:", round(dK, 4))
# for d_K <= 0.01 by the Stein route we need (2/pi)^(1/4) sqrt(3/sqrt(L)) <= 0.01 with L ~ loglog x
print("bound 2 (loglog x)^(-1/4) <= 0.01 needs loglog x >=", f"{200 ** 4:.2e}", " and (2/pi)^(1/4) sqrt(3) =", round((2 / pi) ** 0.25 * sqrt(3), 4), "< 2")
print("half the largest jump of the law (a lower bound for d_K):", round(law.max() / 2, 4), "  1/sigma =", round(1 / sqrt(sig2), 4))

loglog x = 2.6258  mean = 2.8873  variance = 2.4351
Stein: d_W <= 1.6387   d_K <= 1.1435


exact Kolmogorov distance of the model: 0.1504
bound 2 (loglog x)^(-1/4) <= 0.01 needs loglog x >= 1.60e+09  and (2/pi)^(1/4) sqrt(3) = 1.5471 < 2
half the largest jump of the law (a lower bound for d_K): 0.1227   1/sigma = 0.6408


**Problem 1.8 (twelve uniforms).** $W=U_1+\dots+U_{12}-6$: $\mathbb EW^4=2.9$, $\mathbb E\cos W=\bigl(2\sin\frac12\bigr)^{12}$ against $e^{-1/2}$, the bound $\frac1{60}$, and the tail.

In [9]:
EcosW = (2 * math.sin(0.5)) ** 12
print("E cos W =", EcosW, "  exp(-1/2) =", exp(-0.5), "  difference", EcosW - exp(-0.5), "  bound (9/5+3)/(24*12) =", (9 / 5 + 3) / (24 * 12))
W = rng.random((10 ** 6, 12)).sum(axis=1) - 6
print("simulated E W^4:", round(np.mean(W ** 4), 3), " exact 3 - 6/(5*12) =", 3 - 6 / 60)
print("P(|W| > 6) = 0 exactly, while P(|Z| > 6) =", mp.nstr(mp.erfc(6 / mp.sqrt(2)), 4))
# P(W > 4) = P(sum of 12 uniforms > 10) = P(sum < 2) = 2^12/12!
print("P(|W| > 4) = 2(2^12 - 12)/12! =", 2 * (2 ** 12 - 12) / factorial(12), " Irwin-Hall check:", 2 * sum((-1) ** j * comb(12, j) * (2 - j) ** 12 for j in range(3)) / factorial(12), "   P(|Z| > 4) =", mp.nstr(mp.erfc(4 / mp.sqrt(2)), 4))

E cos W = 0.6039679955782548   exp(-1/2) = 0.6065306597126334   difference -0.002562664134378667   bound (9/5+3)/(24*12) = 0.016666666666666666


simulated E W^4: 2.9  exact 3 - 6/(5*12) = 2.9
P(|W| > 6) = 0 exactly, while P(|Z| > 6) = 1.973e-9
P(|W| > 4) = 2(2^12 - 12)/12! = 1.7052135107690662e-05  Irwin-Hall check: 1.7052135107690662e-05    P(|Z| > 4) = 6.334e-5


**Problem 1.9 (random sets without progressions).** In $\Z/N\Z$, $N$ prime, keep each residue with probability $p=cN^{-2/3}$. Simulate $P(\text{no 3-term progression})$
against $e^{-c^3/2}$, and evaluate the Chen--Stein bound.

In [10]:
def no3ap(N, p, reps):
    cnt = 0
    for _ in range(reps):
        A = np.flatnonzero(rng.random(N) < p); s = set(A.tolist()); found = False
        for i in range(len(A)):
            for j in range(i + 1, len(A)):
                a, b = int(A[i]), int(A[j])
                # {a, b} is part of a progression a, b, 2b-a  or  2a-b, a, b  or  a, (a+b)/2, b
                if (2 * b - a) % N in s or (2 * a - b) % N in s or ((a + b) * (N + 1) // 2) % N in s:
                    found = True; break
            if found: break
        cnt += not found
    return cnt / reps
for N, c in ((10007, 1.0), (10007, 1.5), (100003, 1.0)):
    p = c * N ** (-2 / 3); lam = p ** 3 * N * (N - 1) / 2
    print(f"N={N} c={c}: simulated {no3ap(N, p, 1000):.3f}   exp(-lambda)={exp(-lam):.3f}   exp(-c^3/2)={exp(-c**3/2):.3f}")
def bounds(N, c):
    p = c * N ** (-2 / 3); M = N * (N - 1) / 2
    lam = p ** 3 * M
    return (9 * (N - 1) / 2 * (p ** 3 + p ** 2) + 6 * p) * (1 - exp(-lam))     # (b1 + b2)(1 - e^-lam)/lam, see the solution
print("Chen-Stein bound, c=1: N=1e4:", round(bounds(10 ** 4, 1), 3), " N=1e6:", round(bounds(10 ** 6, 1), 4), " N=1e9:", round(bounds(10 ** 9, 1), 5))

N=10007 c=1.0: simulated 0.652   exp(-lambda)=0.607   exp(-c^3/2)=0.607


N=10007 c=1.5: simulated 0.254   exp(-lambda)=0.185   exp(-c^3/2)=0.185


N=100003 c=1.0: simulated 0.630   exp(-lambda)=0.607   exp(-c^3/2)=0.607
Chen-Stein bound, c=1: N=1e4: 0.087  N=1e6: 0.0179  N=1e9: 0.00177


**Problem 1.10 (Spearman's $\rho$).** The normalised rank statistic $W=\sum_ib_ic_{\pi(i)}$: check $\operatorname{Var}W=1$ exactly for small $n$, the conditional
drift $\mathbb E[W'-W\mid\pi]=-\frac2{n-1}W$, and evaluate the bound of the solution.

In [11]:
def spearman_bc(n):
    m = (n + 1) / 2; kappa2 = 12 * sqrt(n - 1) / (n * (n * n - 1))
    b = np.array([(i - m) for i in range(1, n + 1)]) * sqrt(kappa2)
    return b, b.copy()
n = 6
b, c = spearman_bc(n)
Ws = [sum(b[i] * c[p[i]] for i in range(n)) for p in itertools.permutations(range(n))]
print("n=6: mean", round(np.mean(Ws), 12), " variance", round(np.var(Ws), 12))
p = list(rng.permutation(n)); W = sum(b[i] * c[p[i]] for i in range(n))
drift = np.mean([(b[i] * c[p[j]] + b[j] * c[p[i]] - b[i] * c[p[i]] - b[j] * c[p[j]]) for i in range(n) for j in range(n) if i != j])
print("E[W'-W|pi] =", round(drift, 12), "  -2W/(n-1) =", round(-2 * W / (n - 1), 12))
for n in (10, 100, 1000, 10 ** 4, 10 ** 6):
    b, c = spearman_bc(n)
    bound = sqrt((b ** 4).sum() * (c ** 4).sum() / (n - 1)) + 4 / n + 16 * (abs(b) ** 3).sum() * (abs(c) ** 3).sum() / (n - 1)
    print(f"n={n:8d}: bound on d_W = {bound:.4f}   times sqrt(n) = {bound * sqrt(n):.2f}")

n=6: mean -0.0  variance 1.0
E[W'-W|pi] = 0.0255550626   -2W/(n-1) = 0.0255550626
n=      10: bound on d_W = 8.9501   times sqrt(n) = 28.30
n=     100: bound on d_W = 2.9053   times sqrt(n) = 29.05
n=    1000: bound on d_W = 0.9143   times sqrt(n) = 28.91
n=   10000: bound on d_W = 0.2884   times sqrt(n) = 28.84


n= 1000000: bound on d_W = 0.0288   times sqrt(n) = 28.80


**Problem 1.11 (returns of the planar walk).** $G_n=\sum_{t\le n}\binom{2t}t^2/16^t$ against $\frac1\pi\log n$, and the law of the number of returns.

In [12]:
G = 0.0; u = 1.0; out = {}
for t in range(1, 10 ** 7 + 1):
    u *= ((2 * t - 1) / (2 * t)) ** 2; G += u
    if t in (10 ** 3, 10 ** 5, 10 ** 7): out[t] = G
for t, g in out.items():
    print(f"n={t:>9}: G_n = {g:.5f}   log(n)/pi = {log(t)/pi:.5f}   G_n - log(n)/pi = {g - log(t)/pi:.5f}")
# simulate N_n for n = 10^4 double steps
reps, n = 4000, 10 ** 4
Nn = []
for _ in range(reps // 250):
    steps = rng.integers(0, 4, size=(250, 2 * n), dtype=np.int8)
    Xc = np.cumsum((steps == 0).astype(np.int16) - (steps == 1), axis=1)[:, 1::2]
    Yc = np.cumsum((steps == 2).astype(np.int16) - (steps == 3), axis=1)[:, 1::2]
    Nn.append(((Xc == 0) & (Yc == 0)).sum(axis=1))
Nn = np.concatenate(Nn)
g = 0.0; u = 1.0
for t in range(1, n + 1):
    u *= ((2 * t - 1) / (2 * t)) ** 2; g += u
print("n =", n, " E N_n:", round(Nn.mean(), 3), " G_n:", round(g, 3), "  E N_n^2 / (2 G_n^2) =", round((Nn ** 2).mean() / (2 * g * g), 3),
      "  P(N_n > G_n) =", round((Nn > g).mean(), 3), " exp(-1) =", round(exp(-1), 3))

n=     1000: G_n = 2.26532   log(n)/pi = 2.19881   G_n - log(n)/pi = 0.06651
n=   100000: G_n = 3.73096   log(n)/pi = 3.66468   G_n - log(n)/pi = 0.06628
n= 10000000: G_n = 5.19683   log(n)/pi = 5.13055   G_n - log(n)/pi = 0.06628


n = 10000  E N_n: 2.968  G_n: 2.998   E N_n^2 / (2 G_n^2) = 1.126   P(N_n > G_n) = 0.422  exp(-1) = 0.368


**Problem 1.12 (dice modulo seven).** Exact laws of the total of $n$ dice modulo $7,5,4,2$ by convolution, against the formulas of the solution; and the
local limit theorem with factor $3$ for the die with faces $1,4,\dots,16$.

In [13]:
def law_mod(n, q, faces=range(1, 7)):
    L = [Fr(0)] * q; L[0] = Fr(1)
    for _ in range(n):
        M = [Fr(0)] * q
        for r in range(q):
            for f in faces: M[(r + f) % q] += L[r] / 6
        L = M
    return L
print("mod 7:", all(law_mod(n, 7)[0] == Fr(1, 7) + Fr(6, 7) * Fr(-1, 6) ** n for n in range(1, 15)))
print("mod 5:", all(law_mod(n, 5)[r] == Fr(1, 5) + Fr(5 * (r % 5 == n % 5) - 1, 5) / 6 ** n for n in range(1, 15) for r in range(5)))
print("mod 2 uniform:", all(law_mod(n, 2) == [Fr(1, 2)] * 2 for n in range(1, 15)))
for n in (5, 10, 20, 40):
    tv = sum(abs(x - Fr(1, 4)) for x in law_mod(n, 4)) / 2
    print(f"mod 4, n={n}: TV = {float(tv):.3e}   (sqrt2/6)^n = {(sqrt(2) / 6) ** n:.3e}   ratio {float(tv) / (sqrt(2) / 6) ** n:.4f}")
faces = [1, 4, 7, 10, 13, 16]; mu = 8.5; sig = sqrt(9 * 35 / 12)
die = np.zeros(17); die[faces] = 1 / 6; P = np.array([1.0])
for n in range(1, 201):
    P = np.convolve(P, die)
    if n in (10, 50, 200):
        ks = np.arange(len(P)); sel = (ks % 3 == n % 3)
        err = np.max(np.abs(sig * sqrt(n) * P[sel] - 3 * np.exp(-((ks[sel] - n * mu) / (sig * sqrt(n))) ** 2 / 2) / sqrt(2 * pi)))
        print(f"faces 1,4,...,16, n={n}: sup error {err:.5f}, mass off the class {P[~sel].sum():.1e}")

mod 7: True
mod 5: True
mod 2 uniform: True
mod 4, n=5: TV = 5.144e-04   (sqrt2/6)^n = 7.275e-04   ratio 0.7071
mod 4, n=10: TV = 2.646e-07   (sqrt2/6)^n = 5.292e-07   ratio 0.5000
mod 4, n=20: TV = 1.400e-13   (sqrt2/6)^n = 2.801e-13   ratio 0.5000
mod 4, n=40: TV = 3.922e-26   (sqrt2/6)^n = 7.844e-26   ratio 0.5000
faces 1,4,...,16, n=10: sup error 0.01907, mass off the class 0.0e+00
faces 1,4,...,16, n=50: sup error 0.00380, mass off the class 0.0e+00
faces 1,4,...,16, n=200: sup error 0.00095, mass off the class 0.0e+00


**Problem 1.13 (squares modulo $p$).** The law of $a^2+b^2$ modulo $p$: total variation $\frac{p-1}{p^2}$ from uniform, and the bound
$\frac12\sqrt{p-1}\,p^{-k/2}$ after $k$ steps.

In [14]:
for p in (7, 13, 101, 1009):
    Q = np.zeros(p); 
    for a in range(p): Q[a * a % p] += 1 / p
    Qk = Q.copy(); res = []
    for k in range(2, 5):
        Qk = np.real(np.fft.ifft(np.fft.fft(Qk) * np.fft.fft(Q))) if k > 2 else np.real(np.fft.ifft(np.fft.fft(Q) ** 2))
        tv = 0.5 * np.abs(Qk - 1 / p).sum(); res.append((k, round(tv, 8), round(0.5 * sqrt(p - 1) * p ** (-k / 2), 8)))
    print(p, " TV after 2 steps:", res[0][1], " (p-1)/p^2 =", round((p - 1) / p ** 2, 8), "  (k, TV, bound):", res)
for p in (101, 1009):
    k = p * p // 10; step = np.zeros(p); step[1] = step[-1] = 0.5
    Qk = np.real(np.fft.ifft(np.fft.fft(step) ** k))
    print(f"+-1 walk, p={p}, k=p^2/10={k}: TV = {0.5 * np.abs(Qk - 1 / p).sum():.4f}   Fourier lower bound |cos(2pi/p)|^k/2 = {abs(math.cos(2 * pi / p)) ** k / 2:.4f}   exp(-pi^2/5)/2 = {exp(-pi ** 2 / 5) / 2:.4f}")

7  TV after 2 steps: 0.12244898  (p-1)/p^2 = 0.12244898   (k, TV, bound): [(2, np.float64(0.12244898), 0.17496355), (3, np.float64(0.06122449), 0.06613001), (4, np.float64(0.01749271), 0.02499479)]
13  TV after 2 steps: 0.07100592  (p-1)/p^2 = 0.07100592   (k, TV, bound): [(2, np.float64(0.07100592), 0.13323468), (3, np.float64(0.03550296), 0.03695265), (4, np.float64(0.00546199), 0.01024882)]
101  TV after 2 steps: 0.00980296  (p-1)/p^2 = 0.00980296   (k, TV, bound): [(2, np.float64(0.00980296), 0.04950495), (3, np.float64(0.00490148), 0.00492593), (4, np.float64(9.706e-05), 0.00049015)]
1009  TV after 2 steps: 0.0009901  (p-1)/p^2 = 0.0009901   (k, TV, bound): [(2, np.float64(0.0009901), 0.01573291), (3, np.float64(0.00049505), 0.00049529), (4, np.float64(9.8e-07), 1.559e-05)]
+-1 walk, p=101, k=p^2/10=1020: TV = 0.3961   Fourier lower bound |cos(2pi/p)|^k/2 = 0.0694   exp(-pi^2/5)/2 = 0.0695
+-1 walk, p=1009, k=p^2/10=101808: TV = 0.3961   Fourier lower bound |cos(2pi/p)|^k/2 = 0.06

**Problem 1.14 (lucky tickets).** The exact counts, the local limit approximation $10^{2m}/\sqrt{33\pi m}$, and the corrected one with the fourth cumulant.

In [15]:
def lucky(m):
    P = np.array([1], dtype=object)
    for _ in range(m): P = np.convolve(P, np.ones(10, dtype=object))
    return int(sum(x * x for x in P))
k4 = 2 * (-(10 ** 4 - 1) / 120); s2 = 33 / 2
for m in (1, 2, 3, 4, 6, 10):
    L = lucky(m); a = 10 ** (2 * m) / sqrt(33 * pi * m); b = a * (1 + k4 / (8 * s2 ** 2 * m))
    print(f"m={m:2d}: exact {L}  LLT {a:.6g} (ratio {L/a:.5f})  with correction {b:.6g} (ratio {L/b:.6f})")

m= 1: exact 10  LLT 9.82128 (ratio 1.01820)  with correction 9.0698 (ratio 1.102560)
m= 2: exact 670  LLT 694.469 (ratio 0.96477)  with correction 667.901 (ratio 1.003143)
m= 3: exact 55252  LLT 56703.2 (ratio 0.97441)  with correction 55257 (ratio 0.999910)
m= 4: exact 4816030  LLT 4.91064e+06 (ratio 0.98073)  with correction 4.81671e+06 (ratio 0.999860)
m= 6: exact 39581170420  LLT 4.00952e+10 (ratio 0.98718)  with correction 3.95839e+10 (ratio 0.999931)
m=10: exact 3081918923741896840  LLT 3.10576e+18 (ratio 0.99232)  with correction 3.082e+18 (ratio 0.999974)


**Problem 1.15 (signs that cancel).** The number $Z(n)$ of choices of signs with $\pm1\pm2\cdots\pm n=0$, against $\sqrt{6/\pi}\,2^nn^{-3/2}$ when $n\equiv0,3\pmod 4$.

In [16]:
def Z(n):
    c = {0: 1}
    for k in range(1, n + 1):
        d = {}
        for s, v in c.items():
            d[s + k] = d.get(s + k, 0) + v; d[s - k] = d.get(s - k, 0) + v
        c = d
    return c.get(0, 0)
for n in (3, 4, 7, 8, 20, 40, 100, 200):
    z = Z(n); a = sqrt(6 / pi) * 2 ** n * n ** -1.5
    print(n, z, f"  asymptotic {a:.6g}   ratio {z / a:.5f}")
print("first values:", [Z(n) for n in range(1, 13)])

3 2   asymptotic 2.12769   ratio 0.93999
4 2   asymptotic 2.76395   ratio 0.72360
7 8   asymptotic 9.55132   ratio 0.83758
8 14   asymptotic 15.6353   ratio 0.89541
20 15272   asymptotic 16201.5   ratio 0.94263
40 5830034720   asymptotic 6.00635e+09   ratio 0.97065
100 1731024005948725016633786324   asymptotic 1.75186e+27   ratio 0.98810


200 780463610226751719065842218999070243255558586796769387244   asymptotic 7.85154e+56   ratio 0.99403
first values: [0, 0, 2, 2, 0, 0, 8, 14, 0, 0, 70, 124]


**Problem 1.16 (Smirnov's two-sample statistic).** The reflection formula for paths in a strip, checked by brute force, the limit
$1-2\sum_{k\ge1}(-1)^{k-1}e^{-2k^2x^2}$, and the sequence $(k\sqrt2)$ against random points.

In [17]:
def strip_count(n, h):        # paths of length 2n from 0 to 0 with |S| < h, by dynamic programming
    c = {0: 1}
    for _ in range(2 * n):
        d = {}
        for s, v in c.items():
            for t in (s - 1, s + 1):
                if abs(t) < h: d[t] = d.get(t, 0) + v
        c = d
    return c.get(0, 0)
def reflection(n, h):
    return sum((-1) ** abs(k) * comb(2 * n, n + k * h) for k in range(-(n // h) - 1, n // h + 2) if 0 <= n + k * h <= 2 * n)
print("strip counts agree:", all(strip_count(n, h) == reflection(n, h) for n in range(1, 13) for h in range(1, 9)))
K = lambda x: 1 - 2 * sum((-1) ** (k - 1) * exp(-2 * k * k * x * x) for k in range(1, 60))
for n in (50, 500, 5000):
    for x in (0.5, 1.0, 1.36):
        h = math.ceil(x * sqrt(2 * n))
        print(f"n={n:5d} x={x}: P(sqrt(n/2) D <= x) ~ {float(Fr(reflection(n, h), comb(2 * n, n))):.5f}   limit {K(x):.5f}")
print("K(1.36) =", round(K(1.36), 5), "(the 5% critical value)")
Kt = lambda x: sqrt(2 * pi) / x * sum(exp(-(2 * k - 1) ** 2 * pi ** 2 / (8 * x * x)) for k in range(1, 60))
print("theta identity:", [(x, round(K(x), 12), round(Kt(x), 12)) for x in (0.3, 0.7, 1.0, 1.36, 2.0)])
def ks(xs):
    xs = np.sort(xs); N = len(xs); i = np.arange(1, N + 1)
    return sqrt(N) * max((i / N - xs).max(), (xs - (i - 1) / N).max())
for N in (10 ** 3, 10 ** 4, 10 ** 5):
    print(f"N={N}: sqrt(N) D_N for (k sqrt2): {ks((np.arange(1, N + 1) * sqrt(2)) % 1):.4f}   for random points: {ks(rng.random(N)):.4f}")

strip counts agree: True
n=   50 x=0.5: P(sqrt(n/2) D <= x) ~ 0.03325   limit 0.03605
n=   50 x=1.0: P(sqrt(n/2) D <= x) ~ 0.72809   limit 0.73000
n=   50 x=1.36: P(sqrt(n/2) D <= x) ~ 0.96081   limit 0.95051
n=  500 x=0.5: P(sqrt(n/2) D <= x) ~ 0.03970   limit 0.03605
n=  500 x=1.0: P(sqrt(n/2) D <= x) ~ 0.74239   limit 0.73000
n=  500 x=1.36: P(sqrt(n/2) D <= x) ~ 0.95841   limit 0.95051
n= 5000 x=0.5: P(sqrt(n/2) D <= x) ~ 0.03603   limit 0.03605


n= 5000 x=1.0: P(sqrt(n/2) D <= x) ~ 0.72998   limit 0.73000
n= 5000 x=1.36: P(sqrt(n/2) D <= x) ~ 0.95052   limit 0.95051
K(1.36) = 0.95051 (the 5% critical value)
theta identity: [(0.3, 9.305801e-06, 9.305801e-06), (0.7, 0.28876480497, 0.28876480497), (1.0, 0.730000328323, 0.730000328323), (1.36, 0.950514123245, 0.950514123245), (2.0, 0.999329074744, 0.999329074744)]
N=1000: sqrt(N) D_N for (k sqrt2): 0.0536   for random points: 0.7969
N=10000: sqrt(N) D_N for (k sqrt2): 0.0265   for random points: 0.9013
N=100000: sqrt(N) D_N for (k sqrt2): 0.0100   for random points: 1.0579


**Problem 1.17 (random series over primes).** Variances $\sum_p\frac1{p(\log p)^{2a}}$ grow without bound only for $a\le0$; over the integers, for $a\le\frac12$.

In [18]:
sv = np.ones(10 ** 7, dtype=bool); sv[:2] = False
for i in range(2, int(10 ** 3.5) + 1):
    if sv[i]: sv[i * i::i] = False
ps = np.flatnonzero(sv).astype(float)
for a in (0.0, 0.1, 0.25, 0.5):
    print(f"a={a}: sum over primes p < 1e5, 1e6, 1e7:", [round(float((1 / (ps[ps < N] * np.log(ps[ps < N]) ** (2 * a))).sum()), 4) for N in (1e5, 1e6, 1e7)])

a=0.0: sum over primes p < 1e5, 1e6, 1e7: [2.7053, 2.8873, 3.0414]
a=0.1: sum over primes p < 1e5, 1e6, 1e7: [2.276, 2.3857, 2.4754]
a=0.25: sum over primes p < 1e5, 1e6, 1e7: [1.8672, 1.9185, 1.9584]
a=0.5: sum over primes p < 1e5, 1e6, 1e7: [1.5498, 1.5642, 1.5746]


**Problem 1.18 ($\sum\varepsilon_n/n!$).** The gaps: $\frac2{n!}>2\sum_{k>n}\frac1{k!}$, so distinct sign sequences give distinct sums, and $2^n$ intervals of length
$2\sum_{k>n}\frac1{k!}$ cover the set.

In [19]:
for n in range(1, 12):
    tail = sum(mp.mpf(1) / mp.factorial(k) for k in range(n + 1, n + 40))
    print(n, mp.nstr(2 / mp.factorial(n), 5), ">", mp.nstr(2 * tail, 5), "  sum of s-powers, s=0.1:", mp.nstr(2 ** n * (2 * tail) ** 0.1, 5))

1 2.0 > 1.4366   sum of s-powers, s=0.1: 2.0738
2 1.0 > 0.43656   sum of s-powers, s=0.1: 3.6818
3 0.33333 > 0.10323   sum of s-powers, s=0.1: 6.3749
4 0.083333 > 0.019897   sum of s-powers, s=0.1: 10.814
5 0.016667 > 0.0032303   sum of s-powers, s=0.1: 18.033
6 0.0027778 > 0.00045255   sum of s-powers, s=0.1: 29.631
7 0.00039683 > 5.572e-5   sum of s-powers, s=0.1: 48.063
8 4.9603e-5 > 6.1172e-6   sum of s-powers, s=0.1: 77.072
9 5.5115e-6 > 6.0577e-7   sum of s-powers, s=0.1: 122.32
10 5.5115e-7 > 5.4625e-8   sum of s-powers, s=0.1: 192.33
11 5.0104e-8 > 4.5211e-9   sum of s-powers, s=0.1: 299.82


**Problem 1.19 (a smooth density).** $|\varphi(t)|=\prod_n|\cos(t/n^2)|\le e^{-c\sqrt t}$: compute $-\log|\varphi(t)|/\sqrt t$.

In [20]:
def logphi(t, N=10 ** 6):
    n = np.arange(1, N + 1, dtype=float)
    return np.log(np.abs(np.cos(t / n ** 2))).sum()
for t in (10, 100, 1000, 10 ** 4):
    print(t, round(-logphi(t) / sqrt(t), 4))
print("the solution's constant 1/48 =", round(1 / 48, 4), " (a valid but crude lower bound)")

10 0.5063


100 1.1649
1000 1.0598
10000 1.0074
the solution's constant 1/48 = 0.0208  (a valid but crude lower bound)


**Problem 1.21 (Erdős's golden convolution).** $|\hat\mu(\pi\varphi^k)|$ does not tend to $0$; its limit is $c^2$ with $c=\prod_{j\ge1}|\cos(\pi\varphi^{-j})|$.

In [21]:
mp.mp.dps = 50
ph = (1 + mp.sqrt(5)) / 2
cc = mp.fprod(abs(mp.cos(mp.pi * ph ** -j)) for j in range(1, 200))
def muhat(t, N=400):
    return mp.fprod(mp.cos(t * ph ** -n) for n in range(1, N))
for k in (5, 10, 20, 40):
    print(k, mp.nstr(abs(muhat(mp.pi * ph ** k)), 12))
print("c =", mp.nstr(cc, 12), "  c^2 =", mp.nstr(cc ** 2, 12))
mp.mp.dps = 15

5 0.00706160998888
10 0.00661698511624
20 0.00661349326612
40 0.00661349303534
c = 0.0813233855379   c^2 = 0.00661349303534


**Problem 1.22 (Kac's real zeros).** Kac's density integrated numerically, against $\frac2\pi\log n$ plus Kac's constant, and a simulation.

In [22]:
def kac_density(t, n):
    # (1/pi) sqrt( 1/(1-t^2)^2 - (n+1)^2 t^(2n) / (1 - t^(2n+2))^2 ), stable form for t in (0,1)
    t = mp.mpf(t)
    return mp.re(mp.sqrt(1 / (1 - t * t) ** 2 - (n + 1) ** 2 * t ** (2 * n) / (1 - t ** (2 * n + 2)) ** 2)) / mp.pi
mp.mp.dps = 40
for n in (10, 100, 1000):
    I = 4 * mp.quad(lambda t: kac_density(t, n), [0, 0.5, 0.9, 0.99, 1 - mp.mpf(1) / n, 1])
    print(f"n={n}: E N_n = {mp.nstr(I, 8)}   (2/pi) log n = {2 / pi * log(n):.6f}   difference {mp.nstr(I - 2 / mp.pi * mp.log(n), 6)}")
mp.mp.dps = 15
def real_roots(n, reps=3000):
    tot = 0
    for _ in range(reps):
        r = np.roots(rng.standard_normal(n + 1)); tot += np.sum(np.abs(r.imag) < 1e-9)
    return tot / reps
for n in (10, 100):
    print("simulated mean number of real roots, n =", n, ":", round(real_roots(n), 3))

n=10: E N_n = 2.1502723   (2/pi) log n = 1.465871   difference 0.684401


n=100: E N_n = 3.563789   (2/pi) log n = 2.931742   difference 0.632047


n=1000: E N_n = 5.0239855   (2/pi) log n = 4.397614   difference 0.626372


simulated mean number of real roots, n = 10 : 2.124


simulated mean number of real roots, n = 100 : 3.515


## Part B: laboratory

**Experiment 1 (how Gaussian is Spearman's $\rho$?).** Sample $10^5$ random permutations of size $n=20$ and $n=200$, compute $W$, and estimate the Kolmogorov
distance to $N(0,1)$. Compare with the order $n^{-1/2}$ of Problem 1.10.

**Experiment 2 (Bernoulli convolutions).** Histogram $\sum_{n\le40}\varepsilon_n\lambda^n$ for $\lambda=0.55,0.6,1/\varphi,0.7$. For which $\lambda$ does the
histogram look rough? (Pisot numbers $\frac1\lambda$ give rough pictures.)

**Experiment 3 (Kac polynomials).** For $n=50$, plot the roots of random polynomials with Gaussian coefficients in the complex plane. Where do the non-real roots go?

### Solutions

In [23]:
def W_sp(n, reps):
    m = (n + 1) / 2; b = np.arange(1, n + 1) - m
    s = sqrt((b ** 2).sum() ** 2 / (n - 1))
    P = np.argsort(rng.random((reps, n)), axis=1)
    return (b[None, :] * b[P]).sum(axis=1) / s
for n in (20, 200):
    W = np.sort(W_sp(n, 10 ** 5)); F = np.arange(1, len(W) + 1) / len(W)
    from scipy.stats import norm
    print("n =", n, " Kolmogorov distance ~", round(np.abs(F - norm.cdf(W)).max(), 4), "  1/sqrt(n) =", round(1 / sqrt(n), 4))

n = 20  Kolmogorov distance ~ 0.0087   1/sqrt(n) = 0.2236


n = 200  Kolmogorov distance ~ 0.0023   1/sqrt(n) = 0.0707


In [24]:
for lam in (0.55, 0.6, 1 / phi, 0.7):
    e = rng.choice([-1, 1], size=(200000, 40)); v = e @ lam ** np.arange(1, 41)
    h, _ = np.histogram(v, bins=60, density=True)
    print(f"lambda={lam:.4f}: max density {h.max():.3f}, roughness (mean |jump|) {np.abs(np.diff(h)).mean():.3f}")

lambda=0.5500: max density 0.663, roughness (mean |jump|) 0.095


lambda=0.6000: max density 0.549, roughness (mean |jump|) 0.061
lambda=0.6180: max density 0.498, roughness (mean |jump|) 0.040


lambda=0.7000: max density 0.366, roughness (mean |jump|) 0.013


In [25]:
R = np.concatenate([np.roots(rng.standard_normal(51)) for _ in range(200)])
print("median |root| =", round(float(np.median(np.abs(R))), 4), "  share with ||root| - 1| < 0.1:", round(float(np.mean(np.abs(np.abs(R) - 1) < 0.1)), 3))
print("share of roots in each of 8 angular sectors:", np.round(np.histogram(np.angle(R), bins=8, range=(-pi, pi))[0] / len(R), 3))

median |root| = 1.0003   share with ||root| - 1| < 0.1: 0.807
share of roots in each of 8 angular sectors: [0.109 0.124 0.123 0.11  0.142 0.123 0.124 0.143]
